# STRAT-003: Adaptive Trail Strategy

**Concept:** Use on-chain metrics to TIGHTEN the trailing stop, not as hard exits.

**Entry:** STH-SOPR < 1 (100% bottom detection rate)

**Exit:** Trailing stop that adapts based on market conditions:
- Normal: 25-30% trail
- When MVRV > 2.0 or SOPR > 1.02: Tighten to 15-20%
- When MVRV > 2.5 or SOPR > 1.05: Tighten to 10-15%

This way we:
- Capture the +22% extra upside that hard exits miss
- Protect profits more aggressively when overheated
- Let price action determine actual exit

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("STRAT-003: Adaptive Trail Strategy 🎯")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner')
df = df.sort_index()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Entry signals
sth_entry = df['sopr_sth'] < 1
sth_entry_first = sth_entry & ~sth_entry.shift(1).fillna(False)

print(f"STH-SOPR < 1 entry signals: {sth_entry_first.sum()} ({sth_entry_first.sum()/7:.1f}/year)")

---
## Exit Strategy Functions

In [ ]:
@njit
def exit_simple_trail(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx, trail_pct=0.30, stop_loss=0.25):
    """Baseline: Simple fixed trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price
    return len(price_arr) - 1, price_arr[-1]


@njit
def exit_adaptive_mvrv(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx,
                       trail_normal=0.30, trail_hot=0.20, trail_extreme=0.12,
                       mvrv_hot=2.0, mvrv_extreme=2.5, stop_loss=0.25):
    """Trailing stop tightens based on MVRV"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Adaptive trail based on MVRV
        if mvrv > mvrv_extreme:
            trail = trail_extreme
        elif mvrv > mvrv_hot:
            trail = trail_hot
        else:
            trail = trail_normal
        
        if price <= peak * (1 - trail):
            return j, price
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price
    return len(price_arr) - 1, price_arr[-1]


@njit
def exit_adaptive_sopr(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx,
                       trail_normal=0.30, trail_hot=0.20, trail_extreme=0.12,
                       sopr_hot=1.02, sopr_extreme=1.05, stop_loss=0.25):
    """Trailing stop tightens based on SOPR"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        sopr = sopr_arr[j]
        
        if price > peak:
            peak = price
        
        # Adaptive trail based on SOPR
        if sopr > sopr_extreme:
            trail = trail_extreme
        elif sopr > sopr_hot:
            trail = trail_hot
        else:
            trail = trail_normal
        
        if price <= peak * (1 - trail):
            return j, price
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price
    return len(price_arr) - 1, price_arr[-1]


@njit
def exit_adaptive_combined(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx,
                           trail_normal=0.30, trail_hot=0.20, trail_extreme=0.12,
                           mvrv_hot=2.0, mvrv_extreme=2.5,
                           sopr_hot=1.02, sopr_extreme=1.05, stop_loss=0.25):
    """Trailing stop tightens based on MVRV OR SOPR (whichever is more extreme)"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        sopr = sopr_arr[j]
        
        if price > peak:
            peak = price
        
        # Determine trail level (most restrictive wins)
        trail = trail_normal
        
        if mvrv > mvrv_hot or sopr > sopr_hot:
            trail = trail_hot
        if mvrv > mvrv_extreme or sopr > sopr_extreme:
            trail = trail_extreme
        
        if price <= peak * (1 - trail):
            return j, price
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price
    return len(price_arr) - 1, price_arr[-1]


@njit
def exit_adaptive_sth(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx,
                      trail_normal=0.30, trail_hot=0.20, trail_extreme=0.12,
                      sth_hot=1.02, sth_extreme=1.05, stop_loss=0.25):
    """Trailing stop tightens based on STH-SOPR"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        sth = sth_arr[j]
        
        if price > peak:
            peak = price
        
        if sth > sth_extreme:
            trail = trail_extreme
        elif sth > sth_hot:
            trail = trail_hot
        else:
            trail = trail_normal
        
        if price <= peak * (1 - trail):
            return j, price
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price
    return len(price_arr) - 1, price_arr[-1]

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    """Run backtest with given exit function"""
    price_arr = df['price'].values
    sopr_arr = df['sopr'].values
    sth_arr = df['sopr_sth'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price = exit_func(price_arr, sopr_arr, sth_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'entry_mvrv': mvrv_arr[entry_idx],
            'exit_mvrv': mvrv_arr[exit_idx],
            'entry_sopr': sopr_arr[entry_idx],
            'exit_sopr': sopr_arr[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital):
    """Calculate performance metrics"""
    if len(trades) == 0:
        return None
    
    final_equity = trades['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    win_rate = (trades['net_return'] > 0).mean()
    
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 else 0
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    avg_hold = trades['days_held'].mean()
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'n_trades': len(trades),
        'trades_per_year': len(trades) / years,
        'avg_hold': avg_hold,
        'final_equity': final_equity
    }

---
## Run Strategy Comparison

In [ ]:
print("ADAPTIVE TRAIL STRATEGY COMPARISON")
print("Entry: STH-SOPR < 1")
print("="*140)

strategies = [
    # Baselines
    ('1. Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('2. Simple 25% Trail', exit_simple_trail, {'trail_pct': 0.25}),
    ('3. Simple 20% Trail', exit_simple_trail, {'trail_pct': 0.20}),
    ('4. Simple 15% Trail', exit_simple_trail, {'trail_pct': 0.15}),
    
    # MVRV Adaptive
    ('5. MVRV Adaptive (30/20/12)', exit_adaptive_mvrv, 
     {'trail_normal': 0.30, 'trail_hot': 0.20, 'trail_extreme': 0.12, 'mvrv_hot': 2.0, 'mvrv_extreme': 2.5}),
    ('6. MVRV Adaptive (25/18/10)', exit_adaptive_mvrv, 
     {'trail_normal': 0.25, 'trail_hot': 0.18, 'trail_extreme': 0.10, 'mvrv_hot': 2.0, 'mvrv_extreme': 2.5}),
    ('7. MVRV Adaptive (30/15/10)', exit_adaptive_mvrv, 
     {'trail_normal': 0.30, 'trail_hot': 0.15, 'trail_extreme': 0.10, 'mvrv_hot': 2.0, 'mvrv_extreme': 2.5}),
    
    # SOPR Adaptive
    ('8. SOPR Adaptive (30/20/12)', exit_adaptive_sopr, 
     {'trail_normal': 0.30, 'trail_hot': 0.20, 'trail_extreme': 0.12, 'sopr_hot': 1.02, 'sopr_extreme': 1.05}),
    ('9. SOPR Adaptive (25/18/10)', exit_adaptive_sopr, 
     {'trail_normal': 0.25, 'trail_hot': 0.18, 'trail_extreme': 0.10, 'sopr_hot': 1.02, 'sopr_extreme': 1.05}),
    
    # STH-SOPR Adaptive
    ('10. STH Adaptive (30/20/12)', exit_adaptive_sth, 
     {'trail_normal': 0.30, 'trail_hot': 0.20, 'trail_extreme': 0.12, 'sth_hot': 1.02, 'sth_extreme': 1.05}),
    ('11. STH Adaptive (25/18/10)', exit_adaptive_sth, 
     {'trail_normal': 0.25, 'trail_hot': 0.18, 'trail_extreme': 0.10, 'sth_hot': 1.02, 'sth_extreme': 1.05}),
    
    # Combined Adaptive
    ('12. Combined (30/20/12)', exit_adaptive_combined, 
     {'trail_normal': 0.30, 'trail_hot': 0.20, 'trail_extreme': 0.12}),
    ('13. Combined (25/18/10)', exit_adaptive_combined, 
     {'trail_normal': 0.25, 'trail_hot': 0.18, 'trail_extreme': 0.10}),
    ('14. Combined (30/15/08)', exit_adaptive_combined, 
     {'trail_normal': 0.30, 'trail_hot': 0.15, 'trail_extreme': 0.08}),
]

results = []

print(f"{'Strategy':<35} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'/Yr':>6} {'AvgHold':>8}")
print("-"*140)

for name, func, kwargs in strategies:
    trades = run_backtest(df, sth_entry_first, func, **kwargs)
    m = calc_metrics(trades, 100000)
    
    if m:
        print(f"{name:<35} {m['total_return']*100:>+9.0f}% {m['cagr']*100:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['trades_per_year']:>6.1f} {m['avg_hold']:>7.0f}d")
        results.append({'name': name, 'trades': trades, 'metrics': m})

In [ ]:
# Ranking
for r in results:
    m = r['metrics']
    # Score: balance return, sharpe, and trade frequency
    r['score'] = (m['total_return'] * m['sharpe']) / abs(m['max_dd']) if m['max_dd'] != 0 else 0

by_score = sorted(results, key=lambda x: x['score'], reverse=True)

print("\n" + "="*70)
print("RANKING (Return × Sharpe / MaxDD)")
print("="*70)

for i, r in enumerate(by_score[:7]):
    m = r['metrics']
    print(f"\n{i+1}. {r['name']}")
    print(f"   Return: {m['total_return']*100:+,.0f}% | Sharpe: {m['sharpe']:.2f} | MaxDD: {m['max_dd']*100:.0f}%")
    print(f"   Trades: {m['n_trades']} ({m['trades_per_year']:.1f}/yr) | Win Rate: {m['win_rate']*100:.0f}% | Avg Hold: {m['avg_hold']:.0f}d")

In [ ]:
# Compare best adaptive vs simple
print("\n" + "="*70)
print("ADAPTIVE vs SIMPLE COMPARISON")
print("="*70)

simple_best = [r for r in results if 'Simple' in r['name']][0]
adaptive_best = [r for r in by_score if 'Adaptive' in r['name'] or 'Combined' in r['name']][0]

print(f"\n{'Metric':<20} {'Simple Best':>20} {'Adaptive Best':>20} {'Difference':>15}")
print("-"*75)

for metric, label in [('total_return', 'Return'), ('sharpe', 'Sharpe'), ('max_dd', 'Max DD'), ('win_rate', 'Win Rate'), ('n_trades', 'Trades'), ('avg_hold', 'Avg Hold')]:
    s = simple_best['metrics'][metric]
    a = adaptive_best['metrics'][metric]
    if metric in ['total_return', 'max_dd', 'win_rate']:
        print(f"{label:<20} {s*100:>19.1f}% {a*100:>19.1f}% {(a-s)*100:>+14.1f}%")
    elif metric == 'avg_hold':
        print(f"{label:<20} {s:>19.0f}d {a:>19.0f}d {a-s:>+14.0f}d")
    else:
        print(f"{label:<20} {s:>20.2f} {a:>20.2f} {a-s:>+15.2f}")

print(f"\nSimple: {simple_best['name']}")
print(f"Adaptive: {adaptive_best['name']}")

In [ ]:
# Trade details for best adaptive strategy
print("\n" + "="*120)
print(f"TRADE LOG: {adaptive_best['name']}")
print("="*120)

trades = adaptive_best['trades']
print(f"\n{'Entry':<12} {'Exit':<12} {'Days':>6} {'Entry $':>10} {'Exit $':>10} {'Return':>10} {'MVRV In':>8} {'MVRV Out':>9}")
print("-"*100)

for _, t in trades.iterrows():
    print(f"{t['entry_date'].strftime('%Y-%m-%d'):<12} {t['exit_date'].strftime('%Y-%m-%d'):<12} {t['days_held']:>6} ${t['entry_price']:>9,.0f} ${t['exit_price']:>9,.0f} {t['net_return']*100:>+9.0f}% {t['entry_mvrv']:>8.2f} {t['exit_mvrv']:>9.2f}")

---
## Test with Tighter Base Trails (More Trades)

In [ ]:
print("\nTIGHTER TRAIL VARIANTS (More Active Trading)")
print("="*140)

tight_strategies = [
    # Tighter baselines
    ('Simple 12% Trail', exit_simple_trail, {'trail_pct': 0.12}),
    ('Simple 10% Trail', exit_simple_trail, {'trail_pct': 0.10}),
    ('Simple 8% Trail', exit_simple_trail, {'trail_pct': 0.08}),
    
    # Tighter adaptive
    ('MVRV Adapt (20/12/08)', exit_adaptive_mvrv,
     {'trail_normal': 0.20, 'trail_hot': 0.12, 'trail_extreme': 0.08, 'mvrv_hot': 2.0, 'mvrv_extreme': 2.5}),
    ('MVRV Adapt (15/10/06)', exit_adaptive_mvrv,
     {'trail_normal': 0.15, 'trail_hot': 0.10, 'trail_extreme': 0.06, 'mvrv_hot': 2.0, 'mvrv_extreme': 2.5}),
    ('Combined (20/12/08)', exit_adaptive_combined,
     {'trail_normal': 0.20, 'trail_hot': 0.12, 'trail_extreme': 0.08}),
    ('Combined (15/10/06)', exit_adaptive_combined,
     {'trail_normal': 0.15, 'trail_hot': 0.10, 'trail_extreme': 0.06}),
]

print(f"{'Strategy':<30} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'/Yr':>6} {'AvgHold':>8}")
print("-"*100)

tight_results = []
for name, func, kwargs in tight_strategies:
    trades = run_backtest(df, sth_entry_first, func, **kwargs)
    m = calc_metrics(trades, 100000)
    
    if m:
        print(f"{name:<30} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['trades_per_year']:>6.1f} {m['avg_hold']:>7.0f}d")
        tight_results.append({'name': name, 'trades': trades, 'metrics': m})

---
## Summary

In [ ]:
# Find overall best
all_results = results + tight_results
for r in all_results:
    m = r['metrics']
    r['score'] = (m['total_return'] * m['sharpe']) / abs(m['max_dd']) if m['max_dd'] != 0 else 0

all_sorted = sorted(all_results, key=lambda x: x['score'], reverse=True)

print("\n" + "="*70)
print("STRAT-003 SUMMARY")
print("="*70)

print(f"\n📊 Entry: STH-SOPR < 1")
print(f"   • Signals: {sth_entry_first.sum()} ({sth_entry_first.sum()/7:.1f}/year)")

best = all_sorted[0]
m = best['metrics']

print(f"\n🏆 BEST EXIT: {best['name']}")
print(f"   • Return: {m['total_return']*100:+,.0f}%")
print(f"   • Sharpe: {m['sharpe']:.2f}")
print(f"   • Max DD: {m['max_dd']*100:.0f}%")
print(f"   • Win Rate: {m['win_rate']*100:.0f}%")
print(f"   • Trades: {m['n_trades']} ({m['trades_per_year']:.1f}/year)")
print(f"   • Avg Hold: {m['avg_hold']:.0f} days")

# Most active strategy that still performs well
active = sorted([r for r in all_results if r['metrics']['trades_per_year'] >= 5], 
                key=lambda x: x['score'], reverse=True)

if active:
    best_active = active[0]
    m = best_active['metrics']
    print(f"\n⚡ BEST ACTIVE (>5 trades/yr): {best_active['name']}")
    print(f"   • Return: {m['total_return']*100:+,.0f}%")
    print(f"   • Trades: {m['n_trades']} ({m['trades_per_year']:.1f}/year)")
    print(f"   • Win Rate: {m['win_rate']*100:.0f}%")

In [ ]:
# Plot equity curves
import plotly.graph_objects as go

fig = go.Figure()

# Plot top 5 strategies
colors = ['blue', 'green', 'red', 'purple', 'orange']
for i, r in enumerate(all_sorted[:5]):
    trades = r['trades']
    eq_dates = [trades['entry_date'].iloc[0]] + list(trades['exit_date'])
    eq_vals = [100000] + list(trades['equity'])
    fig.add_trace(go.Scatter(x=eq_dates, y=eq_vals, name=r['name'], line=dict(color=colors[i])))

# Add B&H
bh = 100000 * (df['price'] / df['price'].iloc[0])
fig.add_trace(go.Scatter(x=bh.index, y=bh.values, name='Buy & Hold', line=dict(color='gray', dash='dash')))

fig.update_layout(
    title='STRAT-003 Equity Curves (STH-SOPR Entry + Adaptive Trail)',
    yaxis_title='Equity ($)',
    yaxis_type='log',
    height=500
)
fig.show()